# Clase 199 — FastAPI sirviendo modelos

Construye un servicio FastAPI completo (predict, predict-batch, health, metrics) y lo loadtestea con un cliente sincrónico.

Requiere: `pip install fastapi uvicorn[standard] joblib scikit-learn httpx`. Si querés correr el server en background desde el notebook, levantalo en otra terminal.

## Setup

In [ ]:
import os, shutil, tempfile, joblib
from pathlib import Path
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

WORK = Path(tempfile.gettempdir()) / 'fastapi_demo'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(); os.chdir(WORK)

X, y = load_iris(return_X_y=True)
m = RandomForestClassifier(n_estimators=50, random_state=42).fit(X, y)
joblib.dump(m, 'model.pkl')
print('modelo guardado:', Path('model.pkl').stat().st_size, 'bytes')

## 1. `app.py` — servicio FastAPI

In [ ]:
app_src = '''\
from contextlib import asynccontextmanager
from typing import Annotated
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, field_validator
import joblib, numpy as np, time, logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("app")

@asynccontextmanager
async def lifespan(app: FastAPI):
    log.info("loading model...")
    app.state.model = joblib.load("model.pkl")
    app.state.start = time.time()
    log.info("model loaded.")
    yield
    log.info("shutting down.")

app = FastAPI(title="iris-api", version="1.0.0", lifespan=lifespan)

class IrisIn(BaseModel):
    features: Annotated[list[float], Field(min_length=4, max_length=4)]
    @field_validator("features")
    @classmethod
    def positive(cls, v):
        if any(x < 0 for x in v): raise ValueError("features must be non-negative")
        return v

class IrisOut(BaseModel):
    cls: int
    proba: list[float]

class BatchIn(BaseModel):
    rows: list[list[float]]

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": app.state.model is not None,
            "uptime_s": round(time.time() - app.state.start, 1)}

@app.post("/predict", response_model=IrisOut)
def predict(x: IrisIn):
    arr = np.asarray(x.features).reshape(1, -1)
    cls = int(app.state.model.predict(arr)[0])
    proba = app.state.model.predict_proba(arr)[0].tolist()
    return IrisOut(cls=cls, proba=proba)

@app.post("/predict-batch")
def predict_batch(b: BatchIn):
    arr = np.asarray(b.rows)
    if arr.shape[1] != 4:
        raise HTTPException(422, "each row needs 4 features")
    cls = app.state.model.predict(arr).tolist()
    return {"n": len(cls), "predictions": cls}
'''
Path('app.py').write_text(app_src)
print('app.py escrito —', len(app_src.splitlines()), 'líneas')

## 2. Server en background + smoke test

Levantamos `uvicorn` como subprocess y le pegamos con `httpx`.

In [ ]:
import subprocess, time, httpx, sys
proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', '8765', '--log-level', 'warning'],
    cwd=str(WORK), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
# Esperar a que arranque
for _ in range(30):
    try:
        r = httpx.get('http://127.0.0.1:8765/health', timeout=0.5)
        if r.status_code == 200: break
    except Exception: time.sleep(0.2)
print('health:', r.json())

In [ ]:
# Predict OK
print(httpx.post('http://127.0.0.1:8765/predict', json={'features': [5.1, 3.5, 1.4, 0.2]}).json())

# Predict con valor negativo → 422
r = httpx.post('http://127.0.0.1:8765/predict', json={'features': [-1, 2, 3, 4]})
print(r.status_code, r.json())

# Batch
rows = X[:100].tolist()
r = httpx.post('http://127.0.0.1:8765/predict-batch', json={'rows': rows})
print('batch ok, predicciones:', r.json()['n'])

## 3. Loadtest: 1 batch vs 100 requests individuales

In [ ]:
import time
rows = X[:100].tolist()

t0 = time.perf_counter()
for r in rows:
    httpx.post('http://127.0.0.1:8765/predict', json={'features': r})
t_individual = time.perf_counter() - t0

t0 = time.perf_counter()
httpx.post('http://127.0.0.1:8765/predict-batch', json={'rows': rows})
t_batch = time.perf_counter() - t0

print(f'100 requests individuales: {t_individual * 1000:.1f} ms')
print(f'1 request batch (100):    {t_batch * 1000:.1f} ms')
print(f'speedup batch: {t_individual / t_batch:.1f}x')

In [ ]:
# Cleanup
proc.terminate(); proc.wait(timeout=5)
print('server detenido.')

## Ejercicio guiado

1. Agregá `prometheus-fastapi-instrumentator` y un endpoint `GET /metrics`. Verificá con `curl localhost:8765/metrics | grep http_request`.
2. Convertí `/predict` a `async def` y simulá un I/O-bound con `await asyncio.sleep(0.01)` (un "feature store call"). Compará throughput con 50 clientes concurrentes vs el `def` original.
3. Escribí un `locustfile.py` con 100 users y corré 60 s. Reportá p50/p95/p99.
4. Hardenelo para producción: `FastAPI(docs_url=None, redoc_url=None)`, agregá rate limiting con `slowapi`, y log estructurado JSON con `structlog`.

## Conclusiones

- `lifespan` evita re-cargar el modelo por request — diferencia de 100× en latencia.
- Pydantic v2 valida el input gratis (sin schema custom) y bloquea malos requests con 422.
- Batching reduce overhead de HTTP + permite vectorización numpy.
- Sin healthcheck honesto, K8s no sabe cuándo sacar al pod del LB.